In [38]:
import scanpy as sc
import anndata as ad
from matplotlib.gridspec import GridSpec
import matplotlib.pyplot as plt
from matplotlib_scalebar.scalebar import ScaleBar
from matplotlib.colors import ListedColormap, rgb2hex
import numpy as np
import warnings
import pandas as pd
warnings.filterwarnings('ignore')
from sklearn.metrics import jaccard_score
import seaborn as sns
import matplotlib.pyplot as plt
plt.rcParams['pdf.fonttype'] = 42 

from matplotlib.font_manager import fontManager, FontProperties

fontManager.addfont('/data/work/Arial.ttf')

font = FontProperties(fname='/data/work/Arial.ttf')
font_name = font.get_name()
plt.rcParams['font.family'] = font_name

In [39]:
import matplotlib.pyplot as plt
import seaborn as sns
def corr_plot(corr_matrix, save_name, figsize):


    g = sns.clustermap(
        corr_matrix,
        annot=False,
        cmap='coolwarm',
        square=True,
        linewidths=.5,
        cbar_pos=None,          
        dendrogram_ratio=0.15,  
        figsize=figsize,
        row_cluster=True,
        col_cluster=True,
        yticklabels=True,
        xticklabels = True,
        tree_kws={'linewidths': 1.0}
    )

    g.ax_heatmap.yaxis.tick_left()
    g.ax_heatmap.yaxis.set_label_position('left')

    heatmap_pos = g.ax_heatmap.get_position()
    row_dend_pos = g.ax_row_dendrogram.get_position()

    g.ax_row_dendrogram.set_position([heatmap_pos.x1,
                                      row_dend_pos.y0, 
                                     row_dend_pos.width, row_dend_pos.height])

    g.ax_row_dendrogram.invert_xaxis()

    cbar_ax = g.fig.add_axes([heatmap_pos.x1 + 0.2, 0.5, 0.03, 0.2])  
    cbar_ax.set_title('Correlation', fontsize=12, pad=10)
    g.fig.colorbar(g.ax_heatmap.collections[0], cax=cbar_ax)

    g.ax_heatmap.set_xlabel('Mouse Brain celltype', fontsize=14, fontweight='bold')
    g.ax_heatmap.set_ylabel('Lamprey Brain celltype', fontsize=14, fontweight='bold')

    plt.setp(g.ax_heatmap.get_xticklabels(), rotation=45, ha='right')
    plt.setp(g.ax_heatmap.get_yticklabels(), rotation=0)

    plt.savefig(save_name, bbox_inches = 'tight')
    plt.close()

In [40]:
import plotly.graph_objects as go
import numpy as np

def plot_sankey_from_corr(corr_matrix, threshold=0.3, output_path=None):

    lamprey_labels = list(corr_matrix.index)
    mouse_labels = list(corr_matrix.columns)
    all_labels = lamprey_labels + mouse_labels
 
    label_to_idx = {label: idx for idx, label in enumerate(all_labels)}

    sources = []
    targets = []
    values = []
    colors = []
    
    for i, lamprey_type in enumerate(lamprey_labels):
        for j, mouse_type in enumerate(mouse_labels):
            corr_val = corr_matrix.loc[lamprey_type, mouse_type]
            
            if corr_val > threshold:
                sources.append(label_to_idx[lamprey_type])
                targets.append(label_to_idx[mouse_type])
                values.append(corr_val)
                
                alpha = min(corr_val, 1.0)
                colors.append(f'rgba(31, 119, 180, {alpha})')
    
    node_colors = ['rgba(255, 127, 14, 0.8)'] * len(lamprey_labels) + \
                  ['rgba(44, 160, 44, 0.8)'] * len(mouse_labels)
    
    fig = go.Figure(data=[go.Sankey(
        node=dict(
            pad=15,
            thickness=20,
            line=dict(color="black", width=0.5),
            label=all_labels,
            color=node_colors
        ),
        link=dict(
            source=sources,
            target=targets,
            value=values,
            color=colors
        )
    )])
    
    fig.update_layout(
        title_text="Lamprey-Mouse Cell Type Correlation Sankey",
        font_size=10,
        height=1200,
        width=1000
    )
    
    if output_path:
        if output_path.endswith('.html'):
            fig.write_html(output_path)
        else:
            fig.write_image(output_path)
    
    fig.show()
    return fig




In [41]:
adata = sc.read_h5ad('/data/work/22.fusemap/05.stereoalign/02.scvi/01.olfactory_bulb/scvi_corrected.h5ad')
dic = {
    '1': 'mouse',
    '0': 'lamprey',
}
adata.obs['species'] = [dic[i] for i in adata.obs['species']]
lamprey = adata[adata.obs['species'] == 'lamprey']
mouse = adata[adata.obs['species'] == 'mouse']

In [42]:
csv = pd.read_csv('/data/users/wuhaixu/online/240726_newScviMerge/metadata/lamprey.all.csv', index_col = 'Unnamed: 0')
lamprey_pro = pd.read_csv('/data/work/22.fusemap/01.datas/lamprey_color_spatial.csv', index_col = 'Unnamed: 0')
lamprey_pro_dict = dict(zip(lamprey_pro['spatialCluster'], lamprey_pro['spatialClusterV2']))
csv['spatialClusterV2'] = [lamprey_pro_dict[i] for i in csv['anno_whole_cluster']]


In [46]:
csv[csv['region'] == 'olfactory_bulb']

,area,x,y,region,nCounts_RNA,slices,batch,z,cluster,level1,...,level-1,new_cluster,mergeRegion,nCount_RNA,nFeature_RNA,seuratcluster,anno_whole_cluster,mergeRegion.sim,seuratcluster.1,spatialClusterV2
438-2-2,1406,4297.034139,8423.296586,olfactory_bulb,224,3,2.0,8.4,Fibro.1,Tel,...,Telencephalon,Menin.1,Olfactory_Bulb,220,98,10,Ob-Neu-9,ob,ob-10,Ob-Neu-9
3710-2-2,524,4529.452290,8587.681298,olfactory_bulb,216,3,2.0,8.4,Exc.3,Tel,...,Telencephalon,Exc.Tel.6,Olfactory_Bulb,205,119,10,Ob-Neu-9,ob,ob-10,Ob-Neu-9
4231-2-2,1073,4122.725070,8585.548928,olfactory_bulb,382,3,2.0,8.4,Vasc.1,Tel,...,Telencephalon,Menin.1,Olfactory_Bulb,370,195,10,Ob-Neu-9,ob,ob-10,Ob-Neu-9
9194-2-2,1033,4077.981607,8560.574056,olfactory_bulb,215,3,2.0,8.4,Vasc.1,Tel,...,Telencephalon,Epend.4,Olfactory_Bulb,212,98,10,Ob-Neu-9,ob,ob-10,Ob-Neu-9
10630-2-2,938,4670.327292,8661.922175,olfactory_bulb,212,3,2.0,8.4,Fibro.1,Tel,...,Telencephalon,Epend.4,Olfactory_Bulb,198,105,1,Ob-Neu-2,ob,ob-1,Ob-Neu-2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9495-38-38,462,8944.567100,15070.080087,olfactory_bulb,352,39,38.0,1.2,Inh.Di.2,Tel,...,Telencephalon,Exc.Tel.6,Olfactory_Bulb,349,183,10,Ob-Neu-9,ob,ob-10,Ob-Neu-9
9999-38-38,1263,9187.433096,15362.718923,olfactory_bulb,873,39,38.0,1.2,Exc.3,Tel,...,Telencephalon,Exc.Tel.2,Olfactory_Bulb,865,435,10,Ob-Neu-9,ob,ob-10,Ob-Neu-9
10143-38-38,326,9033.447853,15166.079755,olfactory_bulb,479,39,38.0,1.2,Exc.3,Tel,...,Telencephalon,Exc.Tel.2,Olfactory_Bulb,473,245,12,Ob-Neu-11,ob,ob-12,Ob-Neu-11
10318-38-38,1407,9403.249467,15568.904051,olfactory_bulb,337,39,38.0,1.2,Exc.3,Tel,...,Telencephalon,Imm.2,Olfactory_Bulb,332,183,10,Ob-Neu-9,ob,ob-10,Ob-Neu-9


In [48]:
lamprey.obs

,region,slices,ax,ay,az,celltype,species,n_counts,log_counts,n_genes
3396-4-4,olfactory_bulb,5,3.633693,3.770845,8.0,Pal-Neu-16,lamprey,338.0,5.823046,144
5308-4-4,olfactory_bulb,5,4.519597,3.597578,8.0,Pal-Neu-5,lamprey,364.0,5.897154,172
8214-4-4,olfactory_bulb,5,3.933032,3.718685,8.0,Pal-Neu-6,lamprey,308.0,5.730100,131
11769-4-4,olfactory_bulb,5,4.188544,3.702482,8.0,Pal-Neu-16,lamprey,276.0,5.620401,125
13746-4-4,olfactory_bulb,5,3.535976,3.863776,8.0,Pal-Neu-16,lamprey,290.0,5.669881,124
...,...,...,...,...,...,...,...,...,...,...
60170-15-15,olfactory_bulb,16,4.189477,8.063489,5.8,Str-Neu-5,lamprey,331.0,5.802118,163
60409-15-15,olfactory_bulb,16,4.252656,8.464303,5.8,Str-Neu-1,lamprey,487.0,6.188264,250
61410-15-15,olfactory_bulb,16,4.451509,8.421591,5.8,Str-Neu-2,lamprey,495.0,6.204558,252
75027-15-15,olfactory_bulb,16,4.474121,8.369085,5.8,Str-Neu-5,lamprey,149.0,5.003946,78


In [55]:
adata.obs

,region,slices,ax,ay,az,celltype,species,n_counts,log_counts,n_genes
3396-4-4,olfactory_bulb,5,3.633693,3.770845,8.0,Pal-Neu-16,lamprey,338.0,5.823046,144
5308-4-4,olfactory_bulb,5,4.519597,3.597578,8.0,Pal-Neu-5,lamprey,364.0,5.897154,172
8214-4-4,olfactory_bulb,5,3.933032,3.718685,8.0,Pal-Neu-6,lamprey,308.0,5.730100,131
11769-4-4,olfactory_bulb,5,4.188544,3.702482,8.0,Pal-Neu-16,lamprey,276.0,5.620401,125
13746-4-4,olfactory_bulb,5,3.535976,3.863776,8.0,Pal-Neu-16,lamprey,290.0,5.669881,124
...,...,...,...,...,...,...,...,...,...,...
72068,LOLF-COA-COA1,T304,308.293000,718.182000,590.0,Vascular and leptomeningeal cells,mouse,505.0,6.224558,315
586-23,LOLF-COA-COA1,T309,390.178000,706.448000,645.0,Vascular and leptomeningeal cells,mouse,593.0,6.385194,369
55183-32,LOLF-COA-COA1,T325,234.464000,702.386000,807.0,Olfactory bulb inhibitory neurons,mouse,381.0,5.942799,222
58369-12,LOLF-COA-COA1,T331,229.678000,665.320000,860.0,Olfactory bulb inhibitory neurons,mouse,372.0,5.918894,190


In [4]:




lamprey.obs['celltype'] = csv['spatialClusterV2'].tolist()
lamprey.obs['new_celltype'] = lamprey.obs['celltype'].tolist()

mouse_sc = sc.read_h5ad('/data/work/22.fusemap/01.datas/01.olfactory_bulb/mouse_brain_stereo_3d.h5ad')
inte = list(set(mouse.obs.index.tolist()) & set(mouse_sc.obs.index.tolist()))
mouse = mouse[inte]
mouse_sc = mouse_sc[inte]
mouse.obs['new_celltype'] = mouse_sc.obs['cell_cluster']
adata = ad.concat([lamprey, mouse])

In [6]:
df = pd.DataFrame(adata.obs['celltype'].value_counts())
df = df[df['celltype']>100]
temp = adata[adata.obs['celltype'].isin(df.index.tolist())]
temp.obs['celltype'] = temp.obs['species'].astype(str) +'_'+ temp.obs['celltype'].astype(str)

In [10]:
raw_indexs = [
    'lamprey_Ob-Glia-1',
 'lamprey_Ob-Glia-2',
 'lamprey_Ob-Glia-3',
 'lamprey_Ob-Neu-1',
 'lamprey_Ob-Neu-10',
 'lamprey_Ob-Neu-11',
 'lamprey_Ob-Neu-12',
 'lamprey_Ob-Neu-13',
 'lamprey_Ob-Neu-14',
 'lamprey_Ob-Neu-15',
 'lamprey_Ob-Neu-2',
 'lamprey_Ob-Neu-3',
 'lamprey_Ob-Neu-4',
 'lamprey_Ob-Neu-5',
 'lamprey_Ob-Neu-6',
 'lamprey_Ob-Neu-7',
 'lamprey_Ob-Neu-8',
 'lamprey_Ob-Neu-9',
 'lamprey_Pal-Neu-1',
]
raw_columns = ['mouse_Astrocytes',
 'mouse_Dentate gyrus granule neurons',
 'mouse_Endothelial cells',
 'mouse_Microglia',
 'mouse_Olfactory bulb excitatory neurons',
 'mouse_Olfactory bulb inhibitory neurons',
 'mouse_Oligodendrocyte precursor cells',
 'mouse_Oligodendrocytes',
 'mouse_Telencephalon excitatory neurons',
 'mouse_Telencephalon inhibitory neurons',
 'mouse_Vascular and leptomeningeal cells']
scvi_repr = temp.obsm['aligned_scvi']
cell_types = temp.obs['celltype'].values
df = pd.DataFrame(scvi_repr, index=temp.obs.index)
df['cell_type'] = cell_types
mean_repr = df.groupby('cell_type').mean()
corr_matrix = mean_repr.T.corr('pearson')
corr_matrix = corr_matrix[raw_columns].loc[raw_indexs]
corr_matrix.index = [i.replace('lamprey_', '') for i in corr_matrix.index]
corr_matrix.columns = [i.replace('mouse_', ' ') for i in corr_matrix.columns]
corr_plot(corr_matrix, '/data/work/22.fusemap/05.stereoalign/02.scvi/01.olfactory_bulb/01_20251230/ob_1.pdf', (4,6))

In [13]:
df = pd.DataFrame(adata.obs['new_celltype'].value_counts())
df = df[df['new_celltype']>100]
temp = adata[adata.obs['new_celltype'].isin(df.index.tolist())]
temp.obs['new_celltype'] = temp.obs['species'].astype(str) +'_'+ temp.obs['new_celltype'].astype(str)
set(temp.obs['new_celltype'])

{'lamprey_Ob-Glia-1',
 'lamprey_Ob-Glia-2',
 'lamprey_Ob-Glia-3',
 'lamprey_Ob-Neu-1',
 'lamprey_Ob-Neu-10',
 'lamprey_Ob-Neu-11',
 'lamprey_Ob-Neu-12',
 'lamprey_Ob-Neu-13',
 'lamprey_Ob-Neu-14',
 'lamprey_Ob-Neu-15',
 'lamprey_Ob-Neu-2',
 'lamprey_Ob-Neu-3',
 'lamprey_Ob-Neu-4',
 'lamprey_Ob-Neu-5',
 'lamprey_Ob-Neu-6',
 'lamprey_Ob-Neu-7',
 'lamprey_Ob-Neu-8',
 'lamprey_Ob-Neu-9',
 'lamprey_Pal-Neu-1',
 'mouse_ASC_275',
 'mouse_ASC_276',
 'mouse_ASC_277',
 'mouse_ASC_279',
 'mouse_ASC_280',
 'mouse_ASC_282',
 'mouse_CA1_N_GLU_78',
 'mouse_CA1_N_GLU_80',
 'mouse_DG_N_GLU_93',
 'mouse_EDC_263',
 'mouse_EDC_266',
 'mouse_L2/3_IT_GLU_47',
 'mouse_L2/3_IT_GLU_48',
 'mouse_L2/3_IT_GLU_49',
 'mouse_L4/5_IT_GLU_42',
 'mouse_L4/5_IT_GLU_43',
 'mouse_L4/5_IT_GLU_44',
 'mouse_L4/5_IT_GLU_45',
 'mouse_L5/6_NP_GLU_58',
 'mouse_L5/6_NP_GLU_60',
 'mouse_L5/6_NP_GLU_61',
 'mouse_L5_IT_GLU_41',
 'mouse_L5_PT_GLU_53',
 'mouse_L6_Car3_GLU_56',
 'mouse_L6_Car3_GLU_57',
 'mouse_L6_IT_GLU_50',
 'mouse_L6

In [29]:
raw_indexs = [
    'lamprey_Ob-Glia-1',
 'lamprey_Ob-Glia-2',
 'lamprey_Ob-Glia-3',
 'lamprey_Ob-Neu-1',
 'lamprey_Ob-Neu-10',
 'lamprey_Ob-Neu-11',
 'lamprey_Ob-Neu-12',
 'lamprey_Ob-Neu-13',
 'lamprey_Ob-Neu-14',
 'lamprey_Ob-Neu-15',
 'lamprey_Ob-Neu-2',
 'lamprey_Ob-Neu-3',
 'lamprey_Ob-Neu-4',
 'lamprey_Ob-Neu-5',
 'lamprey_Ob-Neu-6',
 'lamprey_Ob-Neu-7',
 'lamprey_Ob-Neu-8',
 'lamprey_Ob-Neu-9',
 'lamprey_Pal-Neu-1',
]
raw_columns = ['mouse_ASC_275',
 'mouse_ASC_276',
 'mouse_ASC_277',
 'mouse_ASC_279',
 'mouse_ASC_280',
 'mouse_ASC_282',
 'mouse_CA1_N_GLU_78',
 'mouse_CA1_N_GLU_80',
 'mouse_DG_N_GLU_93',
 'mouse_EDC_263',
 'mouse_EDC_266',
 'mouse_L2/3_IT_GLU_47',
 'mouse_L2/3_IT_GLU_48',
 'mouse_L2/3_IT_GLU_49',
 'mouse_L4/5_IT_GLU_42',
 'mouse_L4/5_IT_GLU_43',
 'mouse_L4/5_IT_GLU_44',
 'mouse_L4/5_IT_GLU_45',
 'mouse_L5/6_NP_GLU_58',
 'mouse_L5/6_NP_GLU_60',
 'mouse_L5/6_NP_GLU_61',
 'mouse_L5_IT_GLU_41',
 'mouse_L5_PT_GLU_53',
 'mouse_L6_Car3_GLU_56',
 'mouse_L6_Car3_GLU_57',
 'mouse_L6_IT_GLU_50',
 'mouse_L6_IT_GLU_51',
 'mouse_L6_IT_GLU_52',
 'mouse_L6_N_GLU_62',
 'mouse_L6_N_GLU_63',
 'mouse_L6_N_GLU_64',
 'mouse_L6_N_GLU_65',
 'mouse_L6_N_GLU_68',
 'mouse_L6_N_GLU_69',
 'mouse_MGL_289',
 'mouse_MGL_290',
 'mouse_MGL_291',
 'mouse_MGL_292',
 'mouse_OB_N_GABA_160',
 'mouse_OB_N_GABA_161',
 'mouse_OB_N_GABA_162',
 'mouse_OB_N_GABA_163',
 'mouse_OB_N_GABA_164',
 'mouse_OB_N_GABA_165',
 'mouse_OB_N_GABA_166',
 'mouse_OB_N_GABA_167',
 'mouse_OB_N_GABA_168',
 'mouse_OB_N_GABA_174',
 'mouse_OB_N_GLU_175',
 'mouse_OB_N_GLU_176',
 'mouse_OB_N_GLU_177',
 'mouse_OB_N_GLU_178',
 'mouse_OB_N_GLU_179',
 'mouse_OB_N_GLU_181',
 'mouse_OB_N_GLU_182',
 'mouse_OB_N_GLU_183',
 'mouse_OL_294',
 'mouse_OL_299',
 'mouse_OL_300',
 'mouse_OL_302',
 'mouse_OPC_306',
 'mouse_OPC_307',
 'mouse_TE_N_GABA_PVALB_114',
 'mouse_TE_N_GABA_PVALB_115',
 'mouse_TE_N_GABA_PVALB_116',
 'mouse_TE_N_GABA_PVALB_117',
 'mouse_TE_N_GABA_PVALB_118',
 'mouse_TE_N_GABA_RELN_107',
 'mouse_TE_N_GABA_RELN_109',
 'mouse_TE_N_GABA_SST_119',
 'mouse_TE_N_GABA_SST_120',
 'mouse_TE_N_GABA_SST_121',
 'mouse_TE_N_GABA_SST_122',
 'mouse_TE_N_GABA_SST_123',
 'mouse_TE_N_GABA_SST_124',
 'mouse_TE_N_GABA_SST_126',
 'mouse_TE_N_GABA_SST_128',
 'mouse_TE_N_GABA_SST_129',
 'mouse_TE_N_GABA_VIP_111',
 'mouse_TE_N_GLU_1',
 'mouse_TE_N_GLU_10',
 'mouse_TE_N_GLU_11',
 'mouse_TE_N_GLU_12',
 'mouse_TE_N_GLU_13',
 'mouse_TE_N_GLU_14',
 'mouse_TE_N_GLU_15',
 'mouse_TE_N_GLU_18',
 'mouse_TE_N_GLU_19',
 'mouse_TE_N_GLU_2',
 'mouse_TE_N_GLU_20',
 'mouse_TE_N_GLU_21',
 'mouse_TE_N_GLU_22',
 'mouse_TE_N_GLU_25',
 'mouse_TE_N_GLU_26',
 'mouse_TE_N_GLU_27',
 'mouse_TE_N_GLU_29',
 'mouse_TE_N_GLU_30',
 'mouse_TE_N_GLU_31',
 'mouse_TE_N_GLU_35',
 'mouse_TE_N_GLU_39',
 'mouse_TE_N_GLU_40',
 'mouse_TE_N_GLU_6',
 'mouse_TE_N_GLU_7',
 'mouse_TE_N_GLU_77',
 'mouse_TE_N_GLU_9',
 'mouse_VLMC_267',
 'mouse_VLMC_268',
 'mouse_VLMC_269',
 'mouse_VLMC_270',
 'mouse_VLMC_271',
 'mouse_VLMC_274']
scvi_repr = temp.obsm['aligned_scvi']
cell_types = temp.obs['new_celltype'].values
df = pd.DataFrame(scvi_repr, index=temp.obs.index)
df['cell_type'] = cell_types
mean_repr = df.groupby('cell_type').mean()
corr_matrix = mean_repr.T.corr('pearson')
corr_matrix = corr_matrix[raw_columns].loc[raw_indexs]
corr_matrix.index = [i.replace('lamprey_', '') for i in corr_matrix.index]
corr_matrix.columns = [i.replace('mouse_', ' ') for i in corr_matrix.columns]
corr_plot(corr_matrix, '/data/work/22.fusemap/05.stereoalign/02.scvi/01.olfactory_bulb/01_20251230/ob_2.pdf', (30,6))

In [31]:
raw_indexs = [
    'lamprey_Ob-Glia-1',
 'lamprey_Ob-Glia-2',
 'lamprey_Ob-Glia-3',
 'lamprey_Ob-Neu-1',
 'lamprey_Ob-Neu-10',
 'lamprey_Ob-Neu-11',
 'lamprey_Ob-Neu-12',
 'lamprey_Ob-Neu-13',
 'lamprey_Ob-Neu-14',
 'lamprey_Ob-Neu-15',
 'lamprey_Ob-Neu-2',
 'lamprey_Ob-Neu-3',
 'lamprey_Ob-Neu-4',
 'lamprey_Ob-Neu-5',
 'lamprey_Ob-Neu-6',
 'lamprey_Ob-Neu-7',
 'lamprey_Ob-Neu-8',
 'lamprey_Ob-Neu-9',
 'lamprey_Pal-Neu-1',
]
raw_columns = ['mouse_ASC_275',
 'mouse_ASC_276',
 'mouse_ASC_277',
 'mouse_ASC_279',
 'mouse_ASC_280',
 'mouse_ASC_282',
 'mouse_EDC_263',
 'mouse_EDC_266',

 'mouse_MGL_289',
 'mouse_MGL_290',
 'mouse_MGL_291',
 'mouse_MGL_292',
 'mouse_OB_N_GABA_160',
 'mouse_OB_N_GABA_161',
 'mouse_OB_N_GABA_162',
 'mouse_OB_N_GABA_163',
 'mouse_OB_N_GABA_164',
 'mouse_OB_N_GABA_165',
 'mouse_OB_N_GABA_166',
 'mouse_OB_N_GABA_167',
 'mouse_OB_N_GABA_168',
 'mouse_OB_N_GABA_174',
 'mouse_OB_N_GLU_175',
 'mouse_OB_N_GLU_176',
 'mouse_OB_N_GLU_177',
 'mouse_OB_N_GLU_178',
 'mouse_OB_N_GLU_179',
 'mouse_OB_N_GLU_181',
 'mouse_OB_N_GLU_182',
 'mouse_OB_N_GLU_183',
 'mouse_OL_294',
 'mouse_OL_299',
 'mouse_OL_300',
 'mouse_OL_302',
 'mouse_OPC_306',
 'mouse_OPC_307',
 'mouse_TE_N_GABA_PVALB_114',
 'mouse_TE_N_GABA_PVALB_115',
 'mouse_TE_N_GABA_PVALB_116',
 'mouse_TE_N_GABA_PVALB_117',
 'mouse_TE_N_GABA_PVALB_118',
 'mouse_TE_N_GABA_RELN_107',
 'mouse_TE_N_GABA_RELN_109',
 'mouse_TE_N_GABA_SST_119',
 'mouse_TE_N_GABA_SST_120',
 'mouse_TE_N_GABA_SST_121',
 'mouse_TE_N_GABA_SST_122',
 'mouse_TE_N_GABA_SST_123',
 'mouse_TE_N_GABA_SST_124',
 'mouse_TE_N_GABA_SST_126',
 'mouse_TE_N_GABA_SST_128',
 'mouse_TE_N_GABA_SST_129',
 'mouse_TE_N_GABA_VIP_111',
 'mouse_TE_N_GLU_1',
 'mouse_TE_N_GLU_10',
 'mouse_TE_N_GLU_11',
 'mouse_TE_N_GLU_12',
 'mouse_TE_N_GLU_13',
 'mouse_TE_N_GLU_14',
 'mouse_TE_N_GLU_15',
 'mouse_TE_N_GLU_18',
 'mouse_TE_N_GLU_19',
 'mouse_TE_N_GLU_2',
 'mouse_TE_N_GLU_20',
 'mouse_TE_N_GLU_21',
 'mouse_TE_N_GLU_22',
 'mouse_TE_N_GLU_25',
 'mouse_TE_N_GLU_26',
 'mouse_TE_N_GLU_27',
 'mouse_TE_N_GLU_29',
 'mouse_TE_N_GLU_30',
 'mouse_TE_N_GLU_31',
 'mouse_TE_N_GLU_35',
 'mouse_TE_N_GLU_39',
 'mouse_TE_N_GLU_40',
 'mouse_TE_N_GLU_6',
 'mouse_TE_N_GLU_7',
 'mouse_TE_N_GLU_77',
 'mouse_TE_N_GLU_9',
 'mouse_VLMC_267',
 'mouse_VLMC_268',
 'mouse_VLMC_269',
 'mouse_VLMC_270',
 'mouse_VLMC_271',
 'mouse_VLMC_274']
scvi_repr = temp.obsm['aligned_scvi']
cell_types = temp.obs['new_celltype'].values
df = pd.DataFrame(scvi_repr, index=temp.obs.index)
df['cell_type'] = cell_types
mean_repr = df.groupby('cell_type').mean()
corr_matrix = mean_repr.T.corr('pearson')
corr_matrix = corr_matrix[raw_columns].loc[raw_indexs]
corr_matrix.index = [i.replace('lamprey_', '') for i in corr_matrix.index]
corr_matrix.columns = [i.replace('mouse_', ' ') for i in corr_matrix.columns]
corr_plot(corr_matrix, '/data/work/22.fusemap/05.stereoalign/02.scvi/01.olfactory_bulb/01_20251230/ob_3.pdf', (25,6))